# MentalDistress Phase 15A: SDCNL External Boundary Validation

Zero-training exploratory external boundary transfer for `Suicidal|Depressed` across all ten
frozen MentalRoBERTa seeds. The completed Phase 14 package is a mandatory hard gate.

In [1]:
assert 'Suicidal|Depressed' == 'Suicidal|Depressed'
print("Notebook protocol:", 'Suicidal|Depressed')

Notebook protocol: Suicidal|Depressed


## Embedded deterministic core and frozen configuration

The shared core is embedded directly so the notebook remains standalone on Kaggle. CUDA,
the ten seeds, task pair, source hashes, precision, and output paths are frozen here.

In [2]:
"""Deterministic helpers shared by the two Phase 15 external notebooks."""

from __future__ import annotations

import hashlib
import io
import os
import re
import unicodedata
import zipfile
from pathlib import Path, PurePosixPath

import numpy as np
import pandas as pd
from scipy.stats import binomtest
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.metrics.pairwise import linear_kernel


PLACEHOLDERS = {
    "",
    "[deleted]",
    "[removed]",
    "deleted",
    "removed",
    "nan",
    "none",
    "null",
}
RAW_TEXT_COLUMN_TOKENS = (
    "text",
    "selftext",
    "content",
    "sentence",
    "post_body",
    "norm_",
)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def verify_manifest(root: Path) -> pd.DataFrame:
    root = Path(root)
    manifest_path = root / "sha256_manifest.csv"
    if not manifest_path.is_file():
        raise FileNotFoundError(f"Manifest not found: {manifest_path}")
    manifest = pd.read_csv(manifest_path)
    required = {"path", "sha256", "bytes"}
    if not required.issubset(manifest.columns):
        raise ValueError(f"Manifest columns must include {sorted(required)}")
    if manifest["path"].duplicated().any():
        raise ValueError("Manifest contains duplicate paths")

    expected = set(manifest["path"].astype(str))
    actual = {
        path.relative_to(root).as_posix()
        for path in root.rglob("*")
        if path.is_file() and path != manifest_path
    }
    if expected != actual:
        missing = sorted(expected - actual)
        extra = sorted(actual - expected)
        raise ValueError(f"Manifest membership mismatch; missing={missing}, extra={extra}")

    for row in manifest.itertuples(index=False):
        path = root / str(row.path)
        if path.stat().st_size != int(row.bytes):
            raise ValueError(f"Manifest byte-count mismatch: {row.path}")
        if sha256_file(path) != str(row.sha256):
            raise ValueError(f"Manifest SHA-256 mismatch: {row.path}")
    return manifest.sort_values("path", kind="stable").reset_index(drop=True)


def validate_zip_members(archive_path: Path) -> list[str]:
    archive_path = Path(archive_path)
    with zipfile.ZipFile(archive_path) as archive:
        corrupt = archive.testzip()
        if corrupt is not None:
            raise ValueError(f"ZIP CRC failure: {corrupt}")
        files = []
        for info in archive.infolist():
            name = info.filename
            pure = PurePosixPath(name)
            if pure.is_absolute() or ".." in pure.parts or not pure.parts:
                raise ValueError(f"Unsafe ZIP member: {name}")
            if info.is_dir():
                continue
            files.append(name)
    return sorted(files)


def normalize_for_overlap(text: object) -> str:
    value = unicodedata.normalize("NFKC", str(text)).lower()
    value = "".join(
        character
        if not unicodedata.category(character).startswith(("P", "S"))
        else " "
        for character in value
    )
    return re.sub(r"\s+", " ", value).strip()


def _mapped_label(value: object, label_map: dict[object, str]) -> str | None:
    if value in label_map:
        return label_map[value]
    text = str(value).strip()
    for candidate in (text, text.lower()):
        if candidate in label_map:
            return label_map[candidate]
    try:
        numeric = int(float(text))
    except (TypeError, ValueError):
        return None
    return label_map.get(numeric)


def clean_binary_external(
    frame: pd.DataFrame,
    *,
    text_col: str,
    label_col: str,
    label_map: dict[object, str],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    if text_col not in frame or label_col not in frame:
        raise ValueError(f"Required columns missing: {text_col}, {label_col}")
    work = frame.copy()
    if "source_row" not in work:
        work.insert(0, "source_row", np.arange(len(work), dtype=np.int64))
    if work["source_row"].duplicated().any():
        raise ValueError("source_row must be unique")

    work["text"] = work[text_col].fillna("").astype(str).map(
        lambda value: re.sub(r"\s+", " ", value).strip()
    )
    work["label"] = work[label_col].map(lambda value: _mapped_label(value, label_map))
    work["norm_text"] = work["text"].map(normalize_for_overlap)
    invalid = (
        work["label"].isna()
        | work["norm_text"].isin(PLACEHOLDERS)
        | work["text"].str.strip().eq("")
    )
    valid = work.loc[~invalid].copy()

    group_stats = valid.groupby("norm_text", sort=False).agg(
        rows=("source_row", "size"),
        unique_labels=("label", "nunique"),
    )
    conflict_keys = set(group_stats.index[group_stats["unique_labels"] > 1])
    conflict_mask = valid["norm_text"].isin(conflict_keys)
    without_conflicts = valid.loc[~conflict_mask].copy()
    before_same_label = len(without_conflicts)
    without_conflicts = (
        without_conflicts.sort_values("source_row", kind="stable")
        .drop_duplicates("norm_text", keep="first")
        .reset_index(drop=True)
    )
    if "external_row" not in without_conflicts:
        without_conflicts["external_row"] = without_conflicts["source_row"].astype(int)

    label_order = list(dict.fromkeys(label_map.values()))
    label_to_id = {label: index for index, label in enumerate(label_order)}
    without_conflicts["label_id"] = without_conflicts["label"].map(label_to_id).astype(int)
    keep = [
        "external_row",
        "source_row",
        "text",
        "norm_text",
        "label",
        "label_id",
    ]
    clean = without_conflicts[keep].sort_values("source_row", kind="stable").reset_index(drop=True)
    audit = pd.DataFrame(
        [
            {"stage": "raw_rows", "rows": int(len(work))},
            {
                "stage": "invalid_or_placeholder_rows_removed",
                "rows": int(invalid.sum()),
            },
            {
                "stage": "conflicting_duplicate_rows_removed",
                "rows": int(conflict_mask.sum()),
            },
            {
                "stage": "same_label_duplicate_rows_removed",
                "rows": int(before_same_label - len(without_conflicts)),
            },
            {"stage": "internally_clean_rows", "rows": int(len(clean))},
        ]
    )
    return clean, audit


def _row_id_column(frame: pd.DataFrame) -> str:
    for candidate in ("external_row", "original_row", "source_row"):
        if candidate in frame:
            return candidate
    raise ValueError("Frame needs external_row, original_row, or source_row")


def build_exact_overlap_audit(
    external: pd.DataFrame,
    internal_splits: dict[str, pd.DataFrame],
) -> pd.DataFrame:
    if "text" not in external:
        raise ValueError("External frame requires a text column")
    external_id = _row_id_column(external)
    rows: list[dict[str, object]] = []
    for split_name, internal in internal_splits.items():
        if "text" not in internal:
            raise ValueError(f"Internal split {split_name} requires a text column")
        internal_id = _row_id_column(internal)
        raw_lookup: dict[str, list[object]] = {}
        norm_lookup: dict[str, list[object]] = {}
        for record in internal[[internal_id, "text"]].itertuples(index=False, name=None):
            row_id, raw = record
            raw_value = str(raw)
            raw_lookup.setdefault(raw_value, []).append(row_id)
            norm_lookup.setdefault(normalize_for_overlap(raw_value), []).append(row_id)
        for row_id, raw in external[[external_id, "text"]].itertuples(index=False, name=None):
            raw_value = str(raw)
            normalized = normalize_for_overlap(raw_value)
            raw_ids = raw_lookup.get(raw_value, [])
            norm_ids = norm_lookup.get(normalized, [])
            matched = sorted(set(raw_ids) | set(norm_ids), key=str)
            rows.append(
                {
                    "external_row": row_id,
                    "internal_split": split_name,
                    "internal_row": matched[0] if matched else pd.NA,
                    "raw_exact_overlap": bool(raw_ids),
                    "normalized_exact_overlap": bool(norm_ids),
                    "external_sha256": hashlib.sha256(raw_value.encode("utf-8")).hexdigest(),
                    "normalized_sha256": hashlib.sha256(normalized.encode("utf-8")).hexdigest(),
                }
            )
    return pd.DataFrame(rows)


def build_near_overlap_audit(
    external: pd.DataFrame,
    internal_splits: dict[str, pd.DataFrame],
    *,
    thresholds: tuple[float, ...] = (0.90, 0.95, 0.98),
    min_length_ratio: float = 0.80,
    min_chars: int = 80,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    external_id = _row_id_column(external)
    ext_norm = external["text"].map(normalize_for_overlap).tolist()
    internal_records: list[tuple[str, object, str]] = []
    for split_name, internal in internal_splits.items():
        internal_id = _row_id_column(internal)
        internal_records.extend(
            (split_name, row_id, normalize_for_overlap(raw))
            for row_id, raw in internal[[internal_id, "text"]].itertuples(index=False, name=None)
        )
    if not internal_records:
        raise ValueError("At least one internal row is required for near-overlap auditing")
    internal_norm = [record[2] for record in internal_records]
    vectorizer = TfidfVectorizer(analyzer="char", ngram_range=(3, 5), lowercase=False)
    matrix = vectorizer.fit_transform(internal_norm + ext_norm)
    internal_matrix = matrix[: len(internal_norm)]
    external_matrix = matrix[len(internal_norm) :]
    similarity = linear_kernel(external_matrix, internal_matrix)
    nearest_index = np.asarray(similarity.argmax(axis=1)).ravel()
    nearest_value = similarity[np.arange(len(ext_norm)), nearest_index]

    rows = []
    for position, (row_id, norm, index, score) in enumerate(
        zip(external[external_id], ext_norm, nearest_index, nearest_value, strict=True)
    ):
        split_name, internal_row, internal_value = internal_records[int(index)]
        low = min(len(norm), len(internal_value))
        high = max(len(norm), len(internal_value))
        length_ratio = float(low / high) if high else 1.0
        eligibility = len(norm) >= min_chars and len(internal_value) >= min_chars
        rows.append(
            {
                "external_row": row_id,
                "internal_split": split_name,
                "internal_row": internal_row,
                "nearest_similarity": float(score),
                "length_ratio": length_ratio,
                "external_chars": int(len(norm)),
                "internal_chars": int(len(internal_value)),
                "near_overlap_primary": bool(
                    eligibility and score >= 0.95 and length_ratio >= min_length_ratio
                ),
            }
        )
    row_frame = pd.DataFrame(rows)
    count_rows = []
    for threshold in thresholds:
        mask = (
            (row_frame["nearest_similarity"] >= threshold)
            & (row_frame["length_ratio"] >= min_length_ratio)
            & (row_frame["external_chars"] >= min_chars)
            & (row_frame["internal_chars"] >= min_chars)
        )
        count_rows.append(
            {
                "threshold": float(threshold),
                "length_ratio_threshold": float(min_length_ratio),
                "minimum_characters_each": int(min_chars),
                "rows": int(mask.sum()),
            }
        )
    return row_frame, pd.DataFrame(count_rows)


def freeze_external_set(
    clean: pd.DataFrame,
    exact_audit: pd.DataFrame,
) -> tuple[pd.DataFrame, str]:
    overlap_rows = set(
        exact_audit.loc[
            exact_audit["raw_exact_overlap"] | exact_audit["normalized_exact_overlap"],
            "external_row",
        ]
    )
    frozen = clean.loc[~clean["external_row"].isin(overlap_rows)].copy()
    frozen = frozen.sort_values("external_row", kind="stable").reset_index(drop=True)
    identity_columns = [column for column in ("external_row", "label", "label_id") if column in frozen]
    payload = frozen[identity_columns].to_csv(index=False, lineterminator="\n").encode("utf-8")
    return frozen, hashlib.sha256(payload).hexdigest()


def pair_restricted_predictions(
    logits: np.ndarray,
    pair_ids: tuple[int, int],
) -> tuple[np.ndarray, np.ndarray]:
    selected = np.asarray(logits, dtype=np.float64)[:, list(pair_ids)]
    shifted = selected - selected.max(axis=1, keepdims=True)
    exponent = np.exp(shifted)
    probabilities = exponent / exponent.sum(axis=1, keepdims=True)
    local = probabilities.argmax(axis=1)
    predictions = np.asarray(pair_ids, dtype=np.int64)[local]
    return predictions, probabilities


def binary_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
    }


def paired_stratified_bootstrap(
    y_true: np.ndarray,
    global_pred: np.ndarray,
    specialist_pred: np.ndarray,
    *,
    reps: int = 5000,
    seed: int = 2026,
) -> dict[str, float]:
    y_true = np.asarray(y_true)
    global_pred = np.asarray(global_pred)
    specialist_pred = np.asarray(specialist_pred)
    if not (len(y_true) == len(global_pred) == len(specialist_pred)):
        raise ValueError("Paired arrays must have equal lengths")
    class_indices = [np.flatnonzero(y_true == value) for value in np.unique(y_true)]
    if any(len(indices) == 0 for indices in class_indices):
        raise ValueError("Every class must contain at least one row")
    rng = np.random.default_rng(seed)
    deltas = np.empty(reps, dtype=np.float64)
    for replicate in range(reps):
        sampled = np.concatenate(
            [rng.choice(indices, size=len(indices), replace=True) for indices in class_indices]
        )
        deltas[replicate] = f1_score(
            y_true[sampled], specialist_pred[sampled], average="macro", zero_division=0
        ) - f1_score(
            y_true[sampled], global_pred[sampled], average="macro", zero_division=0
        )
    observed = f1_score(
        y_true, specialist_pred, average="macro", zero_division=0
    ) - f1_score(y_true, global_pred, average="macro", zero_division=0)
    low, high = np.percentile(deltas, [2.5, 97.5])
    return {
        "observed_delta": float(observed),
        "ci_low": float(low),
        "ci_high": float(high),
        "reps": int(reps),
        "bootstrap_seed": int(seed),
    }


def exact_mcnemar_p(
    y_true: np.ndarray,
    global_pred: np.ndarray,
    specialist_pred: np.ndarray,
) -> tuple[int, int, float]:
    global_correct = np.asarray(global_pred) == np.asarray(y_true)
    specialist_correct = np.asarray(specialist_pred) == np.asarray(y_true)
    global_only = int(np.sum(global_correct & ~specialist_correct))
    specialist_only = int(np.sum(~global_correct & specialist_correct))
    discordant = global_only + specialist_only
    p_value = 1.0 if discordant == 0 else float(
        binomtest(min(global_only, specialist_only), discordant, 0.5, alternative="two-sided").pvalue
    )
    return global_only, specialist_only, p_value


def holm_adjust(p_values: np.ndarray) -> np.ndarray:
    values = np.asarray(p_values, dtype=np.float64)
    if values.ndim != 1 or np.any(~np.isfinite(values)) or np.any((values < 0) | (values > 1)):
        raise ValueError("p-values must be a finite one-dimensional array in [0, 1]")
    order = np.argsort(values, kind="stable")
    sorted_values = values[order]
    adjusted_sorted = np.maximum.accumulate(
        (len(values) - np.arange(len(values))) * sorted_values
    )
    adjusted_sorted = np.minimum(adjusted_sorted, 1.0)
    adjusted = np.empty_like(adjusted_sorted)
    adjusted[order] = adjusted_sorted
    return adjusted


def assert_no_raw_text_columns(frame: pd.DataFrame) -> None:
    offenders = [
        column
        for column in frame.columns
        if any(token in str(column).lower() for token in RAW_TEXT_COLUMN_TOKENS)
    ]
    if offenders:
        raise ValueError(f"raw-text-like column(s) found: {offenders}")


def deterministic_zip(source_dir: Path, archive_path: Path) -> str:
    source_dir = Path(source_dir)
    archive_path = Path(archive_path)
    files = sorted(path for path in source_dir.rglob("*") if path.is_file())
    with zipfile.ZipFile(
        archive_path,
        "w",
        compression=zipfile.ZIP_DEFLATED,
        compresslevel=9,
    ) as archive:
        for path in files:
            relative = Path(source_dir.name) / path.relative_to(source_dir)
            info = zipfile.ZipInfo(relative.as_posix(), date_time=(2020, 1, 1, 0, 0, 0))
            info.compress_type = zipfile.ZIP_DEFLATED
            info.external_attr = 0o100644 << 16
            archive.writestr(info, path.read_bytes(), compress_type=zipfile.ZIP_DEFLATED, compresslevel=9)
    return sha256_file(archive_path)


import gc
import json
import os
import platform
import random
import shutil
import sys

import torch
from safetensors.torch import load_file
from transformers import AutoModelForSequenceClassification, AutoTokenizer

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

INPUT_ROOT = Path("/kaggle/input")
WORKING_ROOT = Path("/kaggle/working")
OUTPUT_DIR = WORKING_ROOT / 'phase15a_sdcnl_external_validation_results'
OUTPUT_ZIP = WORKING_ROOT / 'MentalDistress_Phase15A_SDCNL_External_Validation_Results.zip'
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator before running Phase 15.")

DEVICE = torch.device("cuda")
ALL_SEEDS = [13, 42, 77, 123, 211, 340, 512, 1024, 2026, 3407]
LABELS = ["Suicidal", "Depressed", "Anxious", "Frustrated", "Others"]
LABEL_TO_ID = {label: index for index, label in enumerate(LABELS)}
SYSTEMS = ["pair_restricted_global", "pair_specialist"]
PAIR = ('Suicidal', 'Depressed')
PAIR_NAME = 'Suicidal|Depressed'
PAIR_IDS = tuple(LABEL_TO_ID[label] for label in PAIR)
PAIR_SLUG = 'suicidal__depressed'
DATASET_SLUG = 'bymahmud/sdcnl-dataset'
EXTERNAL_FILENAME = 'combined-set.csv'
EXPECTED_EXTERNAL_SOURCE_SHA256 = '8f00c4e55fbde15cd4caba729568d3036ed7b6e438b273e39b1e4d02da18b011'
EXPECTED_PHASE14_ZIP_SHA256 = "4f83c651891650f2f8f9d54dda9088f0e0caf19ef4fb73de91ed1e9893e32e8b"
PHASE14_ROOT_NAME = "phase14_final_locked_test_results"
EXPECTED_PHASE14_MANIFEST_SHA256 = "86b29dea1ab68baee40203f090fc8abf14ea0a1bee51522db419d70e3976ec47"
EXPECTED_PHASE8_ZIP_SHA256 = "6d30115f9da6a1963b2fc74eb1b4c057ecb685ca7ae6dcf0c61d177bb3fa89e1"
PHASE8_ROOT_NAME = "phase8_all_pair_specialists_results"
EXPECTED_PHASE8_MANIFEST_SHA256 = "5fe7f75dba52621ec3e6969efd08651b9e5b34ed9bf32444ba02631222ac43f6"
EXPECTED_TRAIN_SHA256 = "1bac0d2f4e0f4d3da940b08a21bd3c6e18b1d958c57cb263c89a900381572d5b"
EXPECTED_VALIDATION_SHA256 = "88835cd714b02efbd230536c7de3f455e807051a23a1786640b0e8b26b430165"
EXPECTED_TEST_SHA256 = "e3a61197d1d4a57e61269be260570cc5aa5b504275f0370b1c379f0453517270"
EXPECTED_SUMMARY_SHA256 = "9eaad5a54f19a396a4145a6d8b28269cfd855afbdbf9b92020961e98d4b27411"
AUDITED_KAGGLE_REFERENCE_PATHS = {
    "phase8": "/kaggle/input/datasets/sumitnath07/mentaldistress-phase8-all-pair-specialists-results/phase8_all_pair_specialists_results",
    "phase14": "/kaggle/input/datasets/sumitnath07/mentaldistress-phase14-results/phase14_final_locked_test_results",
    "locked_data": "/kaggle/input/datasets/sayemahmedshayeed/mentaldistress-locked-dataset",
    "global_models": "/kaggle/input/datasets/sumitnath07/mentalroberta-10-seeds-output/mental_distress_step4",
    "external": '/kaggle/input/datasets/bymahmud/sdcnl-dataset/combined-set.csv',
}
MAX_LENGTH = 512
INFER_BATCH_SIZE = 16
HEAD_INFER_BATCH_SIZE = 64
PROXY_LIMITATION = 'SDCNL is a semantically aligned binary boundary task.'
COMPLETION_BANNER = 'PHASE 15A COMPLETE'
print("CUDA device:", torch.cuda.get_device_name(0))
print("Configured:", PAIR_NAME, DATASET_SLUG)
print("Seeds:", ALL_SEEDS)

CUDA device: Tesla T4
Configured: Suicidal|Depressed bymahmud/sdcnl-dataset
Seeds: [13, 42, 77, 123, 211, 340, 512, 1024, 2026, 3407]


## Phase 14 completion and upstream integrity gate

This cell executes before external discovery. It accepts either the exact Phase 14 ZIP or
Kaggle's manifest-verified extracted result directory, then verifies the one-time
test-opening record, Phase 8 specialist bank, all three locked splits, ten task heads, and
ten canonical global checkpoints.

In [3]:
def resolve_identical_file(filename, *, parent_fragment=None, expected_sha256=None):
    candidates = sorted(INPUT_ROOT.rglob(filename), key=lambda path: path.as_posix())
    if parent_fragment is not None:
        candidates = [
            path for path in candidates
            if parent_fragment.lower() in path.as_posix().lower()
        ]
    if not candidates:
        raise FileNotFoundError(f"Required Kaggle input not found: {filename}")
    digests = {sha256_file(path) for path in candidates}
    if len(digests) != 1:
        raise ValueError(f"Ambiguous non-identical inputs for {filename}: {candidates}")
    selected = candidates[0]
    if expected_sha256 is not None and sha256_file(selected) != expected_sha256:
        raise ValueError(f"SHA-256 mismatch: {selected}")
    return selected


def extract_verified_archive(archive_path, *, expected_sha256, root_name, cache_name):
    if sha256_file(archive_path) != expected_sha256:
        raise ValueError(f"Unexpected archive SHA-256: {archive_path}")
    validate_zip_members(archive_path)
    cache = WORKING_ROOT / cache_name
    if cache.exists():
        shutil.rmtree(cache)
    cache.mkdir(parents=True)
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(cache)
    root = cache / root_name
    verify_manifest(root)
    return root


def resolve_verified_result_input(
    *,
    root_name,
    archive_name,
    expected_archive_sha256,
    expected_manifest_sha256,
    cache_name,
):
    root_candidates = sorted(
        (
            path for path in INPUT_ROOT.rglob(root_name)
            if path.is_dir()
        ),
        key=lambda path: path.as_posix(),
    )
    if root_candidates:
        manifest_digests = set()
        for candidate in root_candidates:
            manifest_path = candidate / "sha256_manifest.csv"
            if not manifest_path.is_file():
                raise FileNotFoundError(f"Result manifest not found: {manifest_path}")
            manifest_digests.add(sha256_file(manifest_path))
        if manifest_digests != {expected_manifest_sha256}:
            raise ValueError(
                f"Unexpected or conflicting manifests for {root_name}: {manifest_digests}"
            )
        selected_root = root_candidates[0]
        verify_manifest(selected_root)
        return selected_root, "extracted_directory", expected_manifest_sha256

    archive_path = resolve_identical_file(
        archive_name,
        expected_sha256=expected_archive_sha256,
    )
    selected_root = extract_verified_archive(
        archive_path,
        expected_sha256=expected_archive_sha256,
        root_name=root_name,
        cache_name=cache_name,
    )
    observed_manifest_sha256 = sha256_file(selected_root / "sha256_manifest.csv")
    if observed_manifest_sha256 != expected_manifest_sha256:
        raise ValueError(
            f"Manifest SHA-256 mismatch after extracting {archive_name}: "
            f"{observed_manifest_sha256}"
        )
    return selected_root, "archive", expected_archive_sha256


phase14_root, phase14_source_kind, phase14_source_identity = resolve_verified_result_input(
    root_name=PHASE14_ROOT_NAME,
    archive_name="MentalDistress_Phase14_Final_Locked_Test_Results.zip",
    expected_archive_sha256=EXPECTED_PHASE14_ZIP_SHA256,
    expected_manifest_sha256=EXPECTED_PHASE14_MANIFEST_SHA256,
    cache_name="phase15_phase14_gate",
)
execution_summary = json.loads(
    (phase14_root / "phase14_execution_summary.json").read_text()
)
assert execution_summary["status"] == "PASS"
assert execution_summary["preopening_freeze_complete"] is True
assert execution_summary["seed_count"] == 10
assert execution_summary["system_count"] == 3
assert execution_summary["variant_count"] == 3
assert execution_summary["test_accessed"] is True
assert execution_summary["test_open_count"] == 1
assert execution_summary["test_rows"] == 1491
assert execution_summary["prediction_rows"] == 134190
assert execution_summary["validation_reproduction_pass_count"] == 10
structure_audit = pd.read_csv(phase14_root / "phase14_structure_audit.csv")
assert len(structure_audit) == 10 and structure_audit["passed"].astype(bool).all()
phase14_reproduction = pd.read_csv(
    phase14_root / "phase14_validation_reproduction_seedwise.csv"
)
assert len(phase14_reproduction) == 10
assert phase14_reproduction["status"].eq("PASS").all()
opening_record = json.loads(
    (phase14_root / "phase14_test_opening_record.json").read_text()
)
assert opening_record["test_opened"] is True
assert opening_record["test_sha256"] == EXPECTED_TEST_SHA256
preopening_spec = json.loads(
    (phase14_root / "phase14_preopening_spec.json").read_text()
)
assert preopening_spec["model_seeds"] == ALL_SEEDS
phase14_gate = {
    "source_kind": phase14_source_kind,
    "source_identity": phase14_source_identity,
    "used_extracted_directory": phase14_source_kind == "extracted_directory",
    "archive_sha256": (
        phase14_source_identity if phase14_source_kind == "archive" else None
    ),
    "manifest_sha256": sha256_file(phase14_root / "sha256_manifest.csv"),
    "status": execution_summary["status"],
    "test_open_count": execution_summary["test_open_count"],
    "test_sha256": opening_record["test_sha256"],
    "validation_reproduction_pass_count": int(
        phase14_reproduction["status"].eq("PASS").sum()
    ),
}
(OUTPUT_DIR / "phase15_phase14_gate.json").write_text(
    json.dumps(phase14_gate, indent=2, sort_keys=True) + "\n"
)
print("PASS: Phase 14 completion gate")


PHASE8_ROOT, phase8_source_kind, phase8_source_identity = resolve_verified_result_input(
    root_name=PHASE8_ROOT_NAME,
    archive_name="MentalDistress_Phase8_All_Pair_Specialists_Results.zip",
    expected_archive_sha256=EXPECTED_PHASE8_ZIP_SHA256,
    expected_manifest_sha256=EXPECTED_PHASE8_MANIFEST_SHA256,
    cache_name="phase15_phase8_input",
)
phase8_protocol = json.loads((PHASE8_ROOT / "phase8_protocol.json").read_text())
assert phase8_protocol["head_count"] == 100
assert phase8_protocol["model_seeds"] == ALL_SEEDS
assert phase8_protocol["routing_performed"] is False
assert phase8_protocol["test_accessed"] is False

TRAIN_PATH = resolve_identical_file("MentalDistress_train_LOCKED.csv")
VALIDATION_PATH = resolve_identical_file("MentalDistress_validation_LOCKED.csv")
TEST_PATH = resolve_identical_file("MentalDistress_test_LOCKED.csv")
assert sha256_file(TRAIN_PATH) == EXPECTED_TRAIN_SHA256
assert sha256_file(VALIDATION_PATH) == EXPECTED_VALIDATION_SHA256
assert sha256_file(TEST_PATH) == opening_record["test_sha256"] == EXPECTED_TEST_SHA256
train_frame = pd.read_csv(TRAIN_PATH)
validation_frame = pd.read_csv(VALIDATION_PATH)
test_frame = pd.read_csv(TEST_PATH)
for frame, rows in ((train_frame, 6954), (validation_frame, 1490), (test_frame, 1491)):
    assert len(frame) == rows
    assert {"original_row", "text", "label"}.issubset(frame.columns)

summary_path = resolve_identical_file(
    "mentalroberta_full_10seed_summary.csv",
    expected_sha256=EXPECTED_SUMMARY_SHA256,
)
global_summary = pd.read_csv(summary_path)
assert sorted(global_summary["seed"].astype(int).tolist()) == ALL_SEEDS
frozen_runs = {int(item["model_seed"]): item for item in preopening_spec["model_runs"]}
summary_run_ids = {
    int(record.seed): str(record.run_id)
    for record in global_summary.itertuples(index=False)
}
frozen_run_ids = {
    seed: str(record["run_id"])
    for seed, record in frozen_runs.items()
}
assert summary_run_ids == frozen_run_ids
MODEL_DIRS = {}
HEAD_DIRS = {}
for seed in ALL_SEEDS:
    frozen = frozen_runs[seed]
    candidates = sorted(
        path for path in INPUT_ROOT.rglob(frozen["run_id"])
        if path.is_dir()
        and (path / "models/best_model/model.safetensors").is_file()
        and (path / "artifacts/validation_predictions.csv").is_file()
    )
    if not candidates:
        raise FileNotFoundError(f"Canonical global run not found: {frozen['run_id']}")
    run_dir = candidates[0]
    verify_manifest(run_dir / "models")
    verify_manifest(run_dir / "artifacts")
    assert sha256_file(run_dir / "models/sha256_manifest.csv") == frozen["model_manifest_sha256"]
    assert sha256_file(run_dir / "artifacts/sha256_manifest.csv") == frozen["artifact_manifest_sha256"]
    MODEL_DIRS[seed] = run_dir / "models/best_model"
    head_dir = PHASE8_ROOT / "heads" / f"seed_{seed}" / PAIR_SLUG
    metadata = json.loads((head_dir / "metadata.json").read_text())
    assert metadata["status"] == "PASS"
    assert metadata["model_seed"] == seed
    assert metadata["pair"] == PAIR_NAME
    assert metadata["run_id"] == frozen["run_id"]
    assert metadata["encoder_model_manifest_sha256"] == frozen["model_manifest_sha256"]
    assert metadata["head_sha256"] == sha256_file(head_dir / "head.safetensors")
    HEAD_DIRS[seed] = head_dir
assert sorted(MODEL_DIRS) == sorted(HEAD_DIRS) == ALL_SEEDS
print("PASS: Phase 8, locked splits, ten global runs, and ten task heads verified")

PASS: Phase 14 completion gate
PASS: Phase 8, locked splits, ten global runs, and ten task heads verified


## Constrained external-source discovery

Only after all internal gates pass may the frozen external dataset resolver be defined.

In [4]:
def resolve_external_path():
    candidates = sorted(
        path for path in INPUT_ROOT.rglob(EXTERNAL_FILENAME)
        if DATASET_SLUG.lower() in path.as_posix().lower()
    )
    if not candidates:
        raise FileNotFoundError(
            f"Attach Kaggle dataset {DATASET_SLUG} containing {EXTERNAL_FILENAME}"
        )
    digests = {sha256_file(path) for path in candidates}
    if digests != {EXPECTED_EXTERNAL_SOURCE_SHA256}:
        raise ValueError(f"External source mismatch or ambiguity: {candidates}")
    return candidates[0]


print("PASS: external resolver initialized after all internal integrity gates")

PASS: external resolver initialized after all internal integrity gates


## External cleaning, three-split contamination audit, and freeze

The source schema and hash are verified, deterministic duplicate rules are applied, and
raw/normalized exact overlap is excluded against train, validation, and test. Character
3-5-gram near overlap is diagnostic and retained. The final row-ID/label set and complete
analysis protocol are hashed before any model is loaded.

In [5]:
EXTERNAL_PATH = resolve_external_path()
resolved_input_preflight = pd.DataFrame(
    [
        {"input": "phase14_results", "path": str(phase14_root), "source_kind": phase14_source_kind, "sha256": phase14_source_identity},
        {"input": "phase8_results", "path": str(PHASE8_ROOT), "source_kind": phase8_source_kind, "sha256": phase8_source_identity},
        {"input": "locked_train", "path": str(TRAIN_PATH), "source_kind": "recursive_file", "sha256": EXPECTED_TRAIN_SHA256},
        {"input": "locked_validation", "path": str(VALIDATION_PATH), "source_kind": "recursive_file", "sha256": EXPECTED_VALIDATION_SHA256},
        {"input": "locked_test", "path": str(TEST_PATH), "source_kind": "recursive_file", "sha256": EXPECTED_TEST_SHA256},
        {"input": "global_summary", "path": str(summary_path), "source_kind": "recursive_file", "sha256": EXPECTED_SUMMARY_SHA256},
        {"input": "external_source", "path": str(EXTERNAL_PATH), "source_kind": "recursive_file", "sha256": EXPECTED_EXTERNAL_SOURCE_SHA256},
    ]
    + [
        {
            "input": f"global_run_seed_{seed}",
            "path": str(MODEL_DIRS[seed].parents[1]),
            "source_kind": "canonical_run_directory",
            "sha256": frozen_runs[seed]["model_manifest_sha256"],
        }
        for seed in ALL_SEEDS
    ]
)
print("Resolved input preflight (before model loading):")
print(resolved_input_preflight.to_string(index=False))
print("Audited reference layouts:")
for input_name, reference_path in AUDITED_KAGGLE_REFERENCE_PATHS.items():
    print(f"  {input_name}: {reference_path}")
resolved_input_preflight.to_csv(
    OUTPUT_DIR / "phase15_resolved_input_preflight.csv", index=False
)
external_source_sha256 = sha256_file(EXTERNAL_PATH)
assert external_source_sha256 == EXPECTED_EXTERNAL_SOURCE_SHA256
external_raw = pd.read_csv(EXTERNAL_PATH)
assert {"selftext", "is_suicide"}.issubset(external_raw.columns)
external_input = pd.DataFrame({
    "source_row": np.arange(len(external_raw), dtype=np.int64),
    "text": external_raw["selftext"],
    "source_label": external_raw["is_suicide"],
})
external_clean, external_cleaning_audit = clean_binary_external(
    external_input,
    text_col="text",
    label_col="source_label",
    label_map={1: "Suicidal", 0: "Depressed"},
)
external_clean["report_label"] = external_clean["label"]
assert set(external_clean["label"]) == {"Suicidal", "Depressed"}
external_clean["label_id"] = external_clean["label"].map(LABEL_TO_ID).astype(int)

internal_splits = {
    "train": train_frame[["original_row", "text"]].copy(),
    "validation": validation_frame[["original_row", "text"]].copy(),
    "test": test_frame[["original_row", "text"]].copy(),
}
NEAR_SIMILARITY_THRESHOLD = 0.95
NEAR_LENGTH_RATIO_THRESHOLD = 0.80
NEAR_MIN_NORMALIZED_CHARS = 80
NEAR_REPORT_THRESHOLDS = (0.90, 0.95, 0.98)

external_exact_overlap = build_exact_overlap_audit(external_clean, internal_splits)
external_near_overlap, external_near_overlap_counts = build_near_overlap_audit(
    external_clean,
    internal_splits,
    thresholds=NEAR_REPORT_THRESHOLDS,
    min_length_ratio=NEAR_LENGTH_RATIO_THRESHOLD,
    min_chars=NEAR_MIN_NORMALIZED_CHARS,
)
external_frozen, final_external_set_sha256 = freeze_external_set(
    external_clean,
    external_exact_overlap,
)
if external_frozen.empty:
    raise ValueError("External set is empty after frozen exact-overlap exclusions")

external_final_identity = external_frozen[
    ["external_row", "source_row", "label", "report_label", "label_id"]
].copy()
for frame in (
    external_cleaning_audit,
    external_exact_overlap,
    external_near_overlap,
    external_near_overlap_counts,
    external_final_identity,
):
    assert_no_raw_text_columns(frame)

external_cleaning_audit.to_csv(OUTPUT_DIR / "external_cleaning_audit.csv", index=False)
external_exact_overlap.to_csv(OUTPUT_DIR / "external_exact_overlap_audit.csv", index=False)
external_near_overlap.to_csv(OUTPUT_DIR / "external_near_overlap_audit.csv", index=False)
external_near_overlap_counts.to_csv(
    OUTPUT_DIR / "external_near_overlap_threshold_counts.csv", index=False
)
external_final_identity.to_csv(OUTPUT_DIR / "external_final_set_no_raw_text.csv", index=False)

external_spec = {
    "phase": "15A" if PAIR_NAME == "Suicidal|Depressed" else "15B",
    "analysis_title": "Exploratory External Boundary Transfer",
    "phase14_source_kind": phase14_source_kind,
    "phase14_source_identity": phase14_source_identity,
    "phase14_expected_archive_sha256": EXPECTED_PHASE14_ZIP_SHA256,
    "phase14_manifest_sha256": EXPECTED_PHASE14_MANIFEST_SHA256,
    "phase8_source_kind": phase8_source_kind,
    "phase8_source_identity": phase8_source_identity,
    "phase8_expected_archive_sha256": EXPECTED_PHASE8_ZIP_SHA256,
    "phase8_manifest_sha256": EXPECTED_PHASE8_MANIFEST_SHA256,
    "external_dataset_slug": DATASET_SLUG,
    "external_filename": EXTERNAL_FILENAME,
    "external_source_sha256": external_source_sha256,
    "final_external_set_sha256": final_external_set_sha256,
    "final_external_rows": int(len(external_frozen)),
    "model_seeds": ALL_SEEDS,
    "pair": PAIR_NAME,
    "systems": SYSTEMS,
    "external_training": False,
    "specialist_retraining": False,
    "routing_performed": False,
    "tau_used": False,
    "model_selection": False,
    "maximum_length": MAX_LENGTH,
    "precision": "float32",
    "automatic_mixed_precision": False,
    "exact_overlap_splits": ["train", "validation", "test"],
    "near_similarity_threshold": NEAR_SIMILARITY_THRESHOLD,
    "near_length_ratio_threshold": NEAR_LENGTH_RATIO_THRESHOLD,
    "near_minimum_characters_each": NEAR_MIN_NORMALIZED_CHARS,
    "near_overlap_excluded": False,
    "historical_three_seed_results_observed": True,
    "retain_all_ten_seed_outcomes": True,
}
external_spec_path = OUTPUT_DIR / "phase15_external_spec.json"
external_spec_path.write_text(json.dumps(external_spec, indent=2, sort_keys=True) + "\n")
external_spec_sha256 = sha256_file(external_spec_path)
(OUTPUT_DIR / "phase15_external_spec.sha256").write_text(external_spec_sha256 + "\n")
print("External source:", EXTERNAL_PATH)
print("Final external rows:", len(external_frozen))
print("Final external-set SHA-256:", final_external_set_sha256)
print("PASS: external set and protocol frozen before model loading")

Resolved input preflight (before model loading):
               input                                                                                                                                   path             source_kind                                                           sha256
     phase14_results                                    /kaggle/input/datasets/sumitnath07/mentaldistress-phase14-results/phase14_final_locked_test_results     extracted_directory 86b29dea1ab68baee40203f090fc8abf14ea0a1bee51522db419d70e3976ec47
      phase8_results              /kaggle/input/datasets/sumitnath07/mentaldistress-phase8-all-pair-specialists-results/phase8_all_pair_specialists_results     extracted_directory 5fe7f75dba52621ec3e6969efd08651b9e5b34ed9bf32444ba02631222ac43f6
        locked_train                                 /kaggle/input/datasets/sayemahmedshayeed/mentaldistress-locked-dataset/MentalDistress_train_LOCKED.csv          recursive_file 1bac0d2f4e0f4d3da940b08a21bd3c6e18b1

## Ten-seed validation reproduction and frozen external inference

For each seed, the required Phase 8 head is reproduced on its saved validation subset
before external inference. The global pair restriction and the specialist consume the
same frozen encoder pass; no gradient, tuning, routing threshold, or checkpoint write is
permitted.

In [6]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=False)


class FrozenPairHead(torch.nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.classifier = torch.nn.Linear(hidden_size, 2)

    def forward(self, features):
        return self.classifier(features)


def encode_texts(model, tokenizer, texts):
    feature_batches = []
    logit_batches = []
    with torch.inference_mode():
        for start in range(0, len(texts), INFER_BATCH_SIZE):
            encoded = tokenizer(
                texts[start : start + INFER_BATCH_SIZE],
                padding=True,
                truncation=True,
                max_length=MAX_LENGTH,
                return_tensors="pt",
            )
            encoded = {name: tensor.to(DEVICE) for name, tensor in encoded.items()}
            output = model(
                **encoded,
                output_hidden_states=True,
                return_dict=True,
            )
            feature_batches.append(output.hidden_states[-1][:, 0, :].float().cpu())
            logit_batches.append(output.logits.float().cpu().numpy())
    return torch.cat(feature_batches, dim=0), np.vstack(logit_batches)


def load_task_head(seed, feature_dimension):
    head_dir = HEAD_DIRS[seed]
    metadata = json.loads((head_dir / "metadata.json").read_text())
    tensors = load_file(str(head_dir / "head.safetensors"), device="cpu")
    assert set(tensors) == {"classifier.weight", "classifier.bias"}
    assert tuple(tensors["classifier.weight"].shape) == (2, feature_dimension)
    assert tuple(tensors["classifier.bias"].shape) == (2,)
    assert metadata["class_mapping"][metadata["class_a"]] == 0
    assert metadata["class_mapping"][metadata["class_b"]] == 1
    head = FrozenPairHead(feature_dimension).to(DEVICE)
    head.load_state_dict(tensors, strict=True)
    head.eval()
    for parameter in head.parameters():
        parameter.requires_grad_(False)
    local_labels = [
        label for label, local_id in sorted(
            metadata["class_mapping"].items(), key=lambda item: item[1]
        )
    ]
    local_to_global = np.array([LABEL_TO_ID[label] for label in local_labels], dtype=int)
    return metadata, head, local_labels, local_to_global


@torch.inference_mode()
def evaluate_specialist_head(head, features):
    # Phase 8 persisted these heads after GPU evaluation in batches of 64.
    # Replaying that device and batch geometry avoids CPU/GPU GEMM drift.
    logit_batches = []
    for start in range(0, len(features), HEAD_INFER_BATCH_SIZE):
        batch = features[start : start + HEAD_INFER_BATCH_SIZE].to(DEVICE)
        logit_batches.append(head(batch).float().cpu())
    logits = torch.cat(logit_batches, dim=0)
    probabilities = torch.softmax(logits, dim=1)
    return logits.numpy(), probabilities.numpy()


validation_pair_mask = validation_frame["label"].isin(PAIR).to_numpy()
validation_pair_indices = np.flatnonzero(validation_pair_mask)
validation_pair = validation_frame.iloc[validation_pair_indices][
    ["original_row", "text", "label"]
].reset_index(drop=True)
validation_prediction_reproduction_rows = []
external_prediction_frames = []

for seed_index, model_seed in enumerate(ALL_SEEDS, start=1):
    print(f"[{seed_index}/10] seed {model_seed}: validation reproduction + external inference")
    seed_everything(model_seed)
    model_dir = MODEL_DIRS[model_seed]
    tokenizer = AutoTokenizer.from_pretrained(
        model_dir, local_files_only=True, use_fast=True
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        model_dir,
        local_files_only=True,
        output_hidden_states=True,
    )
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    model.eval().to(DEVICE)

    # Phase 8 encoded all 1,490 validation rows in their locked file order and
    # selected the pair afterwards. Preserve those encoder batch boundaries.
    validation_all_features, _ = encode_texts(
        model,
        tokenizer,
        validation_frame["text"].astype(str).tolist(),
    )
    validation_features = validation_all_features[validation_pair_indices]
    metadata, head, local_labels, local_to_global = load_task_head(
        model_seed,
        validation_features.shape[1],
    )
    validation_head_logits, validation_head_probabilities = evaluate_specialist_head(
        head, validation_features
    )
    validation_head_predictions = validation_head_probabilities.argmax(axis=1)
    saved_validation = pd.read_csv(
        HEAD_DIRS[model_seed] / "validation_predictions.csv"
    ).reset_index(drop=True)
    assert saved_validation["original_row"].tolist() == validation_pair["original_row"].tolist()
    prediction_exact = np.array_equal(
        validation_head_predictions,
        saved_validation["predicted_binary_id"].to_numpy(dtype=int),
    )
    saved_logits = saved_validation[
        ["logit_class_a", "logit_class_b"]
    ].to_numpy(dtype=np.float32)
    saved_probabilities = saved_validation[
        ["prob_class_a", "prob_class_b"]
    ].to_numpy(dtype=np.float32)
    logit_max_abs_diff = float(
        np.max(np.abs(validation_head_logits - saved_logits))
    )
    probability_max_abs_diff = float(
        np.max(np.abs(validation_head_probabilities - saved_probabilities))
    )
    if not prediction_exact:
        raise RuntimeError(
            f"Seed {model_seed} Phase 8 specialist predictions did not reproduce"
        )
    if not np.allclose(
        validation_head_logits, saved_logits, rtol=1e-6, atol=1e-5
    ):
        raise RuntimeError(
            f"Seed {model_seed} Phase 8 specialist logits differ; "
            f"max_abs_diff={logit_max_abs_diff:.9g}, required rtol=1e-6 atol=1e-5"
        )
    if probability_max_abs_diff > 1e-6:
        raise RuntimeError(
            f"Seed {model_seed} Phase 8 specialist probabilities differ; "
            f"max_abs_diff={probability_max_abs_diff:.9g}, required <=1e-6"
        )
    validation_prediction_reproduction_rows.append({
        "model_seed": model_seed,
        "pair": PAIR_NAME,
        "rows": int(len(saved_validation)),
        "predictions_exact": bool(prediction_exact),
        "logit_max_abs_diff": logit_max_abs_diff,
        "probability_max_abs_diff": probability_max_abs_diff,
        "status": "PASS",
    })

    external_features, external_global_logits = encode_texts(
        model,
        tokenizer,
        external_frozen["text"].astype(str).tolist(),
    )
    global_pair_pred, global_pair_probabilities = pair_restricted_predictions(
        external_global_logits,
        PAIR_IDS,
    )
    global_top1 = external_global_logits.argmax(axis=1)
    external_head_logits, external_head_probabilities = evaluate_specialist_head(
        head, external_features
    )
    specialist_local_pred = external_head_probabilities.argmax(axis=1)
    specialist_pred = local_to_global[specialist_local_pred]
    base = pd.DataFrame({
        "model_seed": model_seed,
        "external_row": external_frozen["external_row"].to_numpy(),
        "source_row": external_frozen["source_row"].to_numpy(),
        "true_id": external_frozen["label_id"].to_numpy(dtype=int),
        "true_label": external_frozen["label"].to_numpy(),
        "report_label": external_frozen["report_label"].to_numpy(),
        "global_top1": global_top1,
        "global_top1_outside_pair": ~np.isin(global_top1, PAIR_IDS),
        "global_pair_probability_class_a": global_pair_probabilities[:, 0],
        "global_pair_probability_class_b": global_pair_probabilities[:, 1],
        "specialist_probability_local_0": external_head_probabilities[:, 0],
        "specialist_probability_local_1": external_head_probabilities[:, 1],
    })
    global_rows = base.copy()
    global_rows["system_id"] = "pair_restricted_global"
    global_rows["predicted_id"] = global_pair_pred
    global_rows["confidence"] = global_pair_probabilities.max(axis=1)
    specialist_rows = base.copy()
    specialist_rows["system_id"] = "pair_specialist"
    specialist_rows["predicted_id"] = specialist_pred
    specialist_rows["confidence"] = external_head_probabilities.max(axis=1)
    for frame in (global_rows, specialist_rows):
        frame["predicted_label"] = frame["predicted_id"].map(
            {index: label for index, label in enumerate(LABELS)}
        )
    external_prediction_frames.extend([global_rows, specialist_rows])

    del model, tokenizer, head, validation_all_features, validation_features, external_features
    gc.collect()
    torch.cuda.empty_cache()

validation_prediction_reproduction = pd.DataFrame(
    validation_prediction_reproduction_rows
)
assert len(validation_prediction_reproduction) == 10
assert validation_prediction_reproduction["status"].eq("PASS").all()
external_predictions = pd.concat(external_prediction_frames, ignore_index=True)
expected_prediction_rows = len(ALL_SEEDS) * len(SYSTEMS) * len(external_frozen)
assert len(external_predictions) == expected_prediction_rows
assert not external_predictions.duplicated(
    ["model_seed", "system_id", "external_row"]
).any()
assert set(external_predictions["model_seed"]) == set(ALL_SEEDS)
assert set(external_predictions["system_id"]) == set(SYSTEMS)
assert_no_raw_text_columns(validation_prediction_reproduction)
assert_no_raw_text_columns(external_predictions)
validation_prediction_reproduction.to_csv(
    OUTPUT_DIR / "external_validation_prediction_reproduction.csv", index=False
)
external_predictions.to_csv(
    OUTPUT_DIR / "external_predictions_no_raw_text.csv", index=False
)
print("PASS: ten validation reproductions and frozen external inference complete")

[1/10] seed 13: validation reproduction + external inference


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[2/10] seed 42: validation reproduction + external inference


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[3/10] seed 77: validation reproduction + external inference


...

[9/10] seed 2026: validation reproduction + external inference


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[10/10] seed 3407: validation reproduction + external inference


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

PASS: ten validation reproductions and frozen external inference complete


## Seedwise metrics and paired statistical inference

The primary effect is specialist-minus-pair-restricted-global Macro-F1. Each seed receives
5,000 class-stratified paired bootstrap resamples and an exact McNemar test; Holm correction
is applied to the predeclared ten-test task family. Across-seed summaries are descriptive.

In [7]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

BOOTSTRAP_REPS = 5000
BOOTSTRAP_SEED = 2026
ALPHA = 0.05
REPORT_LABELS = (
    ["Depressed_proxy", "Others_proxy"]
    if PAIR_NAME == "Depressed|Others"
    else list(PAIR)
)

metric_rows = []
class_rows = []
confusion_rows = []
correction_rows = []
bootstrap_rows = []
mcnemar_rows = []
label_shift_rows = []

for model_seed in ALL_SEEDS:
    seed_predictions = external_predictions.loc[
        external_predictions["model_seed"] == model_seed
    ]
    system_frames = {}
    for system_id in SYSTEMS:
        frame = seed_predictions.loc[
            seed_predictions["system_id"] == system_id
        ].sort_values("external_row", kind="stable").reset_index(drop=True)
        system_frames[system_id] = frame
        y_true = frame["true_id"].to_numpy(dtype=int)
        y_pred = frame["predicted_id"].to_numpy(dtype=int)
        metric_rows.append({
            "model_seed": model_seed,
            "system_id": system_id,
            **binary_metrics(y_true, y_pred),
        })
        report = classification_report(
            y_true,
            y_pred,
            labels=list(PAIR_IDS),
            target_names=REPORT_LABELS,
            output_dict=True,
            zero_division=0,
        )
        for report_label in REPORT_LABELS:
            values = report[report_label]
            class_rows.append({
                "model_seed": model_seed,
                "system_id": system_id,
                "class_label": report_label,
                "precision": float(values["precision"]),
                "recall": float(values["recall"]),
                "f1": float(values["f1-score"]),
                "support": int(values["support"]),
            })
        matrix = confusion_matrix(y_true, y_pred, labels=list(PAIR_IDS))
        for true_position, true_label in enumerate(REPORT_LABELS):
            for predicted_position, predicted_label in enumerate(REPORT_LABELS):
                confusion_rows.append({
                    "model_seed": model_seed,
                    "system_id": system_id,
                    "true_label": true_label,
                    "predicted_label": predicted_label,
                    "rows": int(matrix[true_position, predicted_position]),
                })

    global_frame = system_frames["pair_restricted_global"]
    specialist_frame = system_frames["pair_specialist"]
    assert global_frame["external_row"].tolist() == specialist_frame["external_row"].tolist()
    y_true = global_frame["true_id"].to_numpy(dtype=int)
    global_pred = global_frame["predicted_id"].to_numpy(dtype=int)
    specialist_pred = specialist_frame["predicted_id"].to_numpy(dtype=int)
    global_correct = global_pred == y_true
    specialist_correct = specialist_pred == y_true
    corrections = int(np.sum(~global_correct & specialist_correct))
    harms = int(np.sum(global_correct & ~specialist_correct))
    correction_rows.append({
        "model_seed": model_seed,
        "corrections": corrections,
        "harms": harms,
        "net_corrections": corrections - harms,
        "disagreements": int(np.sum(global_pred != specialist_pred)),
    })
    bootstrap_result = paired_stratified_bootstrap(
        y_true,
        global_pred,
        specialist_pred,
        reps=BOOTSTRAP_REPS,
        seed=BOOTSTRAP_SEED,
    )
    bootstrap_rows.append({"model_seed": model_seed, **bootstrap_result})
    global_only, specialist_only, p_value = exact_mcnemar_p(
        y_true,
        global_pred,
        specialist_pred,
    )
    mcnemar_rows.append({
        "model_seed": model_seed,
        "global_correct_specialist_wrong": global_only,
        "global_wrong_specialist_correct": specialist_only,
        "discordant_total": global_only + specialist_only,
        "p_value": p_value,
    })
    label_shift_rows.append({
        "model_seed": model_seed,
        "rows": int(len(global_frame)),
        "five_class_top1_outside_pair": int(
            global_frame["global_top1_outside_pair"].sum()
        ),
        "five_class_top1_outside_pair_fraction": float(
            global_frame["global_top1_outside_pair"].mean()
        ),
    })

external_seedwise_metrics = pd.DataFrame(metric_rows)
external_per_class_metrics = pd.DataFrame(class_rows)
external_confusion_matrices = pd.DataFrame(confusion_rows)
external_correction_harm = pd.DataFrame(correction_rows)
external_paired_bootstrap = pd.DataFrame(bootstrap_rows)
external_mcnemar_holm = pd.DataFrame(mcnemar_rows)
external_mcnemar_holm["holm_adjusted_p"] = holm_adjust(
    external_mcnemar_holm["p_value"].to_numpy()
)
external_label_space_shift = pd.DataFrame(label_shift_rows)

summary_rows = []
for system_id in SYSTEMS:
    subset = external_seedwise_metrics.loc[
        external_seedwise_metrics["system_id"] == system_id
    ]
    for metric_name in ("accuracy", "macro_f1", "balanced_accuracy"):
        values = subset[metric_name].to_numpy(dtype=float)
        q1, median, q3 = np.quantile(values, [0.25, 0.50, 0.75])
        summary_rows.append({
            "system_id": system_id,
            "metric": metric_name,
            "count": int(len(values)),
            "mean": float(values.mean()),
            "sd": float(values.std(ddof=1)),
            "median": float(median),
            "q1": float(q1),
            "q3": float(q3),
            "iqr": float(q3 - q1),
            "minimum": float(values.min()),
            "maximum": float(values.max()),
            "positive_fraction": np.nan,
        })
metric_pivot = external_seedwise_metrics.pivot(
    index="model_seed", columns="system_id", values="macro_f1"
)
delta_values = (
    metric_pivot["pair_specialist"] - metric_pivot["pair_restricted_global"]
).to_numpy()
q1, median, q3 = np.quantile(delta_values, [0.25, 0.50, 0.75])
summary_rows.append({
    "system_id": "pair_specialist_minus_global",
    "metric": "macro_f1_delta",
    "count": int(len(delta_values)),
    "mean": float(delta_values.mean()),
    "sd": float(delta_values.std(ddof=1)),
    "median": float(median),
    "q1": float(q1),
    "q3": float(q3),
    "iqr": float(q3 - q1),
    "minimum": float(delta_values.min()),
    "maximum": float(delta_values.max()),
    "positive_fraction": float(np.mean(delta_values > 0)),
})
external_equal_seed_summary = pd.DataFrame(summary_rows)

external_claim_guardrails = external_paired_bootstrap.merge(
    external_mcnemar_holm[["model_seed", "holm_adjusted_p"]],
    on="model_seed",
    validate="one_to_one",
)
external_claim_guardrails["interval_direction"] = np.select(
    [
        external_claim_guardrails["ci_low"] > 0,
        external_claim_guardrails["ci_high"] < 0,
    ],
    ["positive", "negative"],
    default="includes_zero",
)
external_claim_guardrails["directional_improvement_claim_supported"] = (
    (external_claim_guardrails["interval_direction"] == "positive")
    & (external_claim_guardrails["holm_adjusted_p"] < ALPHA)
)

output_frames = {
    "external_seedwise_metrics.csv": external_seedwise_metrics,
    "external_equal_seed_summary.csv": external_equal_seed_summary,
    "external_per_class_metrics.csv": external_per_class_metrics,
    "external_confusion_matrices.csv": external_confusion_matrices,
    "external_correction_harm.csv": external_correction_harm,
    "external_paired_bootstrap.csv": external_paired_bootstrap,
    "external_mcnemar_holm.csv": external_mcnemar_holm,
    "external_claim_guardrails.csv": external_claim_guardrails,
    "external_label_space_shift.csv": external_label_space_shift,
}
for filename, frame in output_frames.items():
    assert_no_raw_text_columns(frame)
    frame.to_csv(OUTPUT_DIR / filename, index=False)

print(external_equal_seed_summary.to_string(index=False))
print("PASS: metrics, paired bootstrap, McNemar, Holm, and claim guardrails complete")

                   system_id            metric  count      mean       sd    median        q1       q3      iqr   minimum  maximum  positive_fraction
      pair_restricted_global          accuracy     10  0.706446 0.004549  0.706061  0.702479 0.709780 0.007300  0.701377 0.714601                NaN
      pair_restricted_global          macro_f1     10  0.702888 0.006063  0.702176  0.697981 0.708360 0.010380  0.695364 0.712045                NaN
      pair_restricted_global balanced_accuracy     10  0.706503 0.004539  0.706092  0.702547 0.709827 0.007280  0.701454 0.714653                NaN
             pair_specialist          accuracy     10  0.702865 0.007089  0.705785  0.701791 0.706749 0.004959  0.687603 0.711295                NaN
             pair_specialist          macro_f1     10  0.700055 0.007993  0.702525  0.698050 0.704246 0.006197  0.683917 0.710660                NaN
             pair_specialist balanced_accuracy     10  0.702917 0.007081  0.705842  0.701852 0.706789 0.00

## Reviewer figures, report, manifest, and deterministic result ZIP

All figures and tables retain every seed outcome. The report uses the title
**Exploratory External Boundary Transfer**, applies task-specific semantic guardrails, and
packages only no-text artifacts after exact manifest verification.

In [8]:
FIGURE_DIR = OUTPUT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
plt.style.use("seaborn-v0_8-whitegrid")

fig, ax = plt.subplots(figsize=(9, 5))
for system_id, marker in zip(SYSTEMS, ("o", "s"), strict=True):
    frame = external_seedwise_metrics.loc[
        external_seedwise_metrics["system_id"] == system_id
    ].sort_values("model_seed")
    ax.plot(frame["model_seed"].astype(str), frame["macro_f1"], marker=marker, label=system_id)
ax.set_xlabel("Model seed")
ax.set_ylabel("Macro-F1")
ax.set_title(f"{PAIR_NAME}: zero-training external Macro-F1")
ax.legend()
fig.tight_layout()
for extension in ("png", "svg"):
    fig.savefig(FIGURE_DIR / f"external_seedwise_macro_f1.{extension}", dpi=180)
plt.close(fig)

delta_frame = metric_pivot.reset_index()
delta_frame["delta"] = delta_frame["pair_specialist"] - delta_frame["pair_restricted_global"]
fig, ax = plt.subplots(figsize=(9, 5))
colors = np.where(delta_frame["delta"] >= 0, "#2a9d8f", "#e76f51")
ax.bar(delta_frame["model_seed"].astype(str), delta_frame["delta"], color=colors)
ax.axhline(0, color="black", linewidth=1)
ax.set_xlabel("Model seed")
ax.set_ylabel("Specialist - global Macro-F1")
ax.set_title(f"{PAIR_NAME}: seedwise external transfer delta")
fig.tight_layout()
for extension in ("png", "svg"):
    fig.savefig(FIGURE_DIR / f"external_seedwise_macro_f1_delta.{extension}", dpi=180)
plt.close(fig)

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(external_correction_harm))
ax.bar(x - 0.2, external_correction_harm["corrections"], width=0.4, label="corrections")
ax.bar(x + 0.2, external_correction_harm["harms"], width=0.4, label="harms")
ax.set_xticks(x, external_correction_harm["model_seed"].astype(str))
ax.set_xlabel("Model seed")
ax.set_ylabel("Rows")
ax.set_title(f"{PAIR_NAME}: specialist corrections and harms")
ax.legend()
fig.tight_layout()
for extension in ("png", "svg"):
    fig.savefig(FIGURE_DIR / f"external_corrections_harms.{extension}", dpi=180)
plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for axis, system_id in zip(axes, SYSTEMS, strict=True):
    aggregate = (
        external_confusion_matrices.loc[
            external_confusion_matrices["system_id"] == system_id
        ]
        .groupby(["true_label", "predicted_label"], sort=False)["rows"]
        .mean()
        .unstack(fill_value=0)
        .reindex(index=REPORT_LABELS, columns=REPORT_LABELS)
    )
    image = axis.imshow(aggregate.to_numpy(), cmap="Blues")
    axis.set_xticks(range(2), REPORT_LABELS, rotation=20)
    axis.set_yticks(range(2), REPORT_LABELS)
    axis.set_title(system_id)
    for row in range(2):
        for column in range(2):
            axis.text(column, row, f"{aggregate.iloc[row, column]:.1f}", ha="center", va="center")
fig.suptitle(f"{PAIR_NAME}: mean confusion matrices across seeds")
fig.tight_layout()
for extension in ("png", "svg"):
    fig.savefig(FIGURE_DIR / f"external_mean_confusion_matrices.{extension}", dpi=180)
plt.close(fig)

input_inventory = resolved_input_preflight.copy()
input_inventory.to_csv(OUTPUT_DIR / "phase15_input_inventory.csv", index=False)
environment = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda_runtime": torch.version.cuda,
    "gpu_name": torch.cuda.get_device_name(0),
    "precision": "float32",
    "automatic_mixed_precision": False,
}
(OUTPUT_DIR / "phase15_environment.json").write_text(
    json.dumps(environment, indent=2, sort_keys=True) + "\n"
)
protocol = dict(external_spec)
protocol.update({
    "bootstrap_reps": BOOTSTRAP_REPS,
    "bootstrap_seed": BOOTSTRAP_SEED,
    "bootstrap_method": "class_stratified_paired_rows",
    "mcnemar_method": "exact_two_sided",
    "holm_family_size": 10,
    "primary_effect": "pair_specialist_minus_pair_restricted_global_macro_f1",
})
protocol_path = OUTPUT_DIR / "phase15_protocol.json"
protocol_path.write_text(json.dumps(protocol, indent=2, sort_keys=True) + "\n")
(OUTPUT_DIR / "phase15_protocol.sha256").write_text(sha256_file(protocol_path) + "\n")

delta_summary = external_equal_seed_summary.loc[
    external_equal_seed_summary["metric"] == "macro_f1_delta"
].iloc[0]
report_lines = [
    "# Exploratory External Boundary Transfer",
    "",
    f"Dataset: `{DATASET_SLUG}`",
    f"Boundary: `{PAIR_NAME}`",
    f"Final external rows: {len(external_frozen)}",
    f"Final external-set SHA-256: `{final_external_set_sha256}`",
    "",
    "## Scope",
    "",
    "This is frozen zero-training pair-restricted transfer, not full five-class external evaluation.",
    "External outcomes did not change any internal model, specialist, hard pair, or threshold.",
    PROXY_LIMITATION,
    "",
    "## Equal-seed summary",
    "",
    external_equal_seed_summary.to_csv(index=False).strip(),
    "",
    "## Primary descriptive effect",
    "",
    f"Mean specialist-minus-global Macro-F1: {delta_summary['mean']:.6f}",
    f"Positive-seed fraction: {delta_summary['positive_fraction']:.3f}",
    "All ten positive, null, and negative seed outcomes are retained.",
    "Confidence intervals containing zero do not support directional improvement claims.",
    "",
    "## Label-space shift",
    "",
    external_label_space_shift.to_csv(index=False).strip(),
    "",
    "## Guardrails",
    "",
    "Published benchmark scores are not directly comparable to this zero-training pair-restricted setup.",
    "The results do not establish universal transfer, clinical validity, safety, or deployment readiness.",
]
(OUTPUT_DIR / "external_report.md").write_text("\n".join(report_lines) + "\n")

structure_checks = pd.DataFrame([
    {"check": "phase14_gate", "observed": int(phase14_gate["status"] == "PASS"), "expected": 1, "passed": phase14_gate["status"] == "PASS"},
    {"check": "model_seeds", "observed": len(ALL_SEEDS), "expected": 10, "passed": len(ALL_SEEDS) == 10},
    {"check": "task_heads", "observed": len(HEAD_DIRS), "expected": 10, "passed": len(HEAD_DIRS) == 10},
    {"check": "validation_reproductions", "observed": len(validation_prediction_reproduction), "expected": 10, "passed": len(validation_prediction_reproduction) == 10},
    {"check": "prediction_rows", "observed": len(external_predictions), "expected": expected_prediction_rows, "passed": len(external_predictions) == expected_prediction_rows},
    {"check": "bootstrap_rows", "observed": len(external_paired_bootstrap), "expected": 10, "passed": len(external_paired_bootstrap) == 10},
    {"check": "holm_rows", "observed": len(external_mcnemar_holm), "expected": 10, "passed": len(external_mcnemar_holm) == 10},
    {"check": "raw_text_columns", "observed": 0, "expected": 0, "passed": True},
])
assert structure_checks["passed"].all()
structure_checks.to_csv(OUTPUT_DIR / "phase15_structure_audit.csv", index=False)
execution_summary = {
    "phase": external_spec["phase"],
    "status": "PASS",
    "seed_count": len(ALL_SEEDS),
    "system_count": len(SYSTEMS),
    "external_rows": int(len(external_frozen)),
    "prediction_rows": int(len(external_predictions)),
    "validation_reproduction_pass_count": int(
        validation_prediction_reproduction["status"].eq("PASS").sum()
    ),
    "bootstrap_reps_per_seed": BOOTSTRAP_REPS,
    "bootstrap_rows": int(len(external_paired_bootstrap)),
    "holm_family_size": int(len(external_mcnemar_holm)),
    "external_training": False,
    "tuning_performed": False,
    "routing_performed": False,
}
(OUTPUT_DIR / "phase15_execution_summary.json").write_text(
    json.dumps(execution_summary, indent=2, sort_keys=True) + "\n"
)

for csv_path in OUTPUT_DIR.rglob("*.csv"):
    assert_no_raw_text_columns(pd.read_csv(csv_path, nrows=1))
manifest_path = OUTPUT_DIR / "sha256_manifest.csv"
manifest_rows = []
for path in sorted(
    candidate for candidate in OUTPUT_DIR.rglob("*")
    if candidate.is_file() and candidate != manifest_path
):
    manifest_rows.append({
        "path": path.relative_to(OUTPUT_DIR).as_posix(),
        "sha256": sha256_file(path),
        "bytes": path.stat().st_size,
    })
pd.DataFrame(manifest_rows).to_csv(manifest_path, index=False)
verify_manifest(OUTPUT_DIR)

repeat_zip = WORKING_ROOT / (OUTPUT_ZIP.stem + "_repeat.zip")
first_zip_sha256 = deterministic_zip(OUTPUT_DIR, OUTPUT_ZIP)
second_zip_sha256 = deterministic_zip(OUTPUT_DIR, repeat_zip)
if first_zip_sha256 != second_zip_sha256:
    raise RuntimeError("two deterministic ZIP builds differ")
repeat_zip.unlink()
validate_zip_members(OUTPUT_ZIP)
print(COMPLETION_BANNER)
print("Result ZIP:", OUTPUT_ZIP)
print("Result ZIP SHA-256:", first_zip_sha256)

PHASE 15A COMPLETE
Result ZIP: /kaggle/working/MentalDistress_Phase15A_SDCNL_External_Validation_Results.zip
Result ZIP SHA-256: 8fcb24274ae71cca3af1ecd37f700afbcc7a9b32cb2c05587edb989e38aa7b66
